# LF Weight Trend Diagnostics

This notebook checks whether the luminosity-function completeness correction behaves as expected. The main sanity check is whether high-redshift clusters receive a larger richness boost after applying `LF_WEIGHT`.


In [ ]:
from pathlib import Path
import pickle
import sys

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from astropy.table import Table, join, unique
from scipy.stats import spearmanr, pearsonr

# On NERSC, repo-level imports live under DESI/Projection.
REPO_DIR = Path('/global/homes/z/zzhang13/DESI/Projection')
if REPO_DIR.exists() and str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

try:
    from setup import data_dir
    CATALOG_DIR = Path(data_dir())
except Exception:
    if REPO_DIR.exists():
        CATALOG_DIR = REPO_DIR / 'catalogs'
    else:
        CATALOG_DIR = Path.cwd().parent / 'catalogs'

PLOT_DIR = Path.cwd() / 'diagnostic_plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

MATCHED_LF_FILE = CATALOG_DIR / 'bgs_clus_RM_gal_matched_with_geoFrac_lfweight.pickle'
RM_SPEC_FILE = CATALOG_DIR / 'rm_clusters_with_spec_richness_lfweighted.pickle'

Z_COL = 'Z_SPEC_x'
RM_RICHNESS_COL = 'LAMBDA'
NO_LF_COL = 'lambda_true'
LF_COL = 'lambda_spec_noproj_weighted'

z_bins = [(0.1, 0.2), (0.2, 0.3), (0.3, 0.4)]
z_colors = ['tab:blue', 'tab:orange', 'tab:red']

mpl.rcParams.update({
    'font.size': 12,
    'axes.linewidth': 1.1,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'figure.dpi': 120,
})

print('Catalog dir:', CATALOG_DIR)
print('Matched LF file:', MATCHED_LF_FILE)
print('Cluster richness file:', RM_SPEC_FILE)


In [ ]:
def read_pickle_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    with path.open('rb') as handle:
        obj = pickle.load(handle)
    if isinstance(obj, Table):
        return obj
    if isinstance(obj, pd.DataFrame):
        return Table.from_pandas(obj)
    if hasattr(obj, 'to_pandas'):
        return Table.from_pandas(obj.to_pandas())
    return Table(obj)


def col_float(table, col):
    return np.asarray(table[col], dtype=float)


def first_existing_col(table, candidates, label):
    for col in candidates:
        if col in table.colnames:
            return col
    raise KeyError(f'Could not find {label}. Tried {candidates}. Available columns include: {table.colnames[:30]}')


def finite_positive(x):
    x = np.asarray(x, dtype=float)
    return np.isfinite(x) & (x > 0)


def summarize_by_z(x, z, bins=z_bins):
    rows = []
    x = np.asarray(x, dtype=float)
    z = np.asarray(z, dtype=float)
    for zlo, zhi in bins:
        mask = (z >= zlo) & (z < zhi) & np.isfinite(x)
        if np.count_nonzero(mask) == 0:
            rows.append((zlo, zhi, 0, np.nan, np.nan, np.nan, np.nan))
            continue
        vals = x[mask]
        rows.append((
            zlo,
            zhi,
            len(vals),
            np.nanpercentile(vals, 16),
            np.nanmedian(vals),
            np.nanpercentile(vals, 84),
            np.nanmean(vals),
        ))
    return pd.DataFrame(rows, columns=['zlo', 'zhi', 'N', 'p16', 'median', 'p84', 'mean'])


def style_axis(ax):
    ax.tick_params(direction='in', top=True, right=True)
    for spine in ax.spines.values():
        spine.set_linewidth(1.1)


In [ ]:
bgs_lf = read_pickle_table(MATCHED_LF_FILE)
required_weight_cols = ['LF_WEIGHT', 'lf_observed_fraction', Z_COL, 'ID']
missing = [col for col in required_weight_cols if col not in bgs_lf.colnames]
if missing:
    raise KeyError(f'Missing required columns in {MATCHED_LF_FILE}: {missing}')

# The LF correction is cluster-level. Collapse the matched galaxy table to one row per ID
# before diagnosing the LF-weight distribution.
cluster_lf_catalog = unique(bgs_lf[['ID', Z_COL, 'LF_WEIGHT', 'lf_observed_fraction']], keys='ID')

LF_WEIGHT = col_float(cluster_lf_catalog, 'LF_WEIGHT')
lf_obs_frac = col_float(cluster_lf_catalog, 'lf_observed_fraction')
z_gal = col_float(cluster_lf_catalog, Z_COL)

valid_weight = finite_positive(LF_WEIGHT) & np.isfinite(z_gal)
print(f'Matched galaxy rows: {len(bgs_lf):,}')
print(f'Unique clusters: {len(cluster_lf_catalog):,}')
print(f'Valid cluster LF weights: {np.count_nonzero(valid_weight):,}')
print('LF weight percentiles:', np.nanpercentile(LF_WEIGHT[valid_weight], [0, 1, 16, 50, 84, 95, 99, 99.5, 100]))
print('Observed fraction percentiles:', np.nanpercentile(lf_obs_frac[np.isfinite(lf_obs_frac)], [0, 1, 16, 50, 84, 95, 99, 100]))

display(summarize_by_z(LF_WEIGHT[valid_weight], z_gal[valid_weight]))


## LF Weight Distribution

If the LF correction is driving the redshift correction, the high-redshift bins should show a tail toward larger `w_LF`.


In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.8))
vals = LF_WEIGHT[valid_weight]
upper = np.nanpercentile(vals, 99.5)
bins = np.logspace(np.log10(max(np.nanmin(vals), 1.0)), np.log10(upper), 70)

ax.hist(vals, bins=bins, histtype='step', lw=1.8, color='black')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'$w_{\rm LF}$')
ax.set_ylabel('Number of clusters')
ax.axvline(np.nanmedian(vals), color='black', ls='--', lw=1.2, label='median')
ax.legend(frameon=False)
style_axis(ax)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'LF_WEIGHT_distribution_global.png', dpi=180)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.8))
vals = LF_WEIGHT[valid_weight]
upper = np.nanpercentile(vals, 99.5)
bins = np.logspace(np.log10(max(np.nanmin(vals), 1.0)), np.log10(upper), 70)

for (zlo, zhi), color in zip(z_bins, z_colors):
    mask = valid_weight & (z_gal >= zlo) & (z_gal < zhi)
    ax.hist(
        LF_WEIGHT[mask],
        bins=bins,
        histtype='step',
        lw=1.8,
        color=color,
        density=True,
        label=rf'${zlo}<z<{zhi}$, $N={np.count_nonzero(mask):,}$',
    )

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'$w_{\rm LF}$')
ax.set_ylabel('Normalized density')
ax.legend(frameon=False, fontsize=10)
style_axis(ax)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'LF_WEIGHT_distribution_by_redshift.png', dpi=180)
plt.show()


## LF Weight Versus Redshift

This plot shows the galaxy-level correction as a function of the cluster redshift used for weighting. The points show binned medians and the 16--84 percentile range.


In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.8))
plot_mask = valid_weight

ax.scatter(z_gal[plot_mask], LF_WEIGHT[plot_mask], s=3, alpha=0.05, color='0.25', rasterized=True)

z_grid_bins = np.linspace(0.1, 0.4, 16)
z_centers = 0.5 * (z_grid_bins[:-1] + z_grid_bins[1:])
med, p16, p84, nbin = [], [], [], []
for zlo, zhi in zip(z_grid_bins[:-1], z_grid_bins[1:]):
    mask = plot_mask & (z_gal >= zlo) & (z_gal < zhi)
    nbin.append(np.count_nonzero(mask))
    if np.count_nonzero(mask) < 10:
        med.append(np.nan); p16.append(np.nan); p84.append(np.nan)
    else:
        vals = LF_WEIGHT[mask]
        p16.append(np.nanpercentile(vals, 16))
        med.append(np.nanmedian(vals))
        p84.append(np.nanpercentile(vals, 84))
med = np.asarray(med)
p16 = np.asarray(p16)
p84 = np.asarray(p84)

ax.errorbar(
    z_centers,
    med,
    yerr=[med - p16, p84 - med],
    fmt='o',
    color='tab:red',
    ecolor='tab:red',
    capsize=0,
    label='median and 16-84 percentile',
)
ax.set_yscale('log')
ax.set_xlabel(r'$z_{\rm spec,BCG}$')
ax.set_ylabel(r'$w_{\rm LF}$')
ax.legend(frameon=False)
style_axis(ax)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'LF_WEIGHT_vs_redshift.png', dpi=180)
plt.show()


## Cluster Richness Before and After LF Weighting

This section compares the old cluster-level spectroscopic richness against the LF-weighted cluster-level richness. This is the key diagnostic for whether high-redshift clusters are boosted.


In [ ]:
rm_spec = read_pickle_table(RM_SPEC_FILE)

missing = [col for col in ['ID', NO_LF_COL, LF_COL] if col not in rm_spec.colnames]
if missing:
    raise KeyError(f'Missing required columns in {RM_SPEC_FILE}: {missing}')

comparison = rm_spec.copy()

# Expected cluster-level LF boost from the LF-weighted matched catalog.
cluster_LF_WEIGHT = unique(cluster_lf_catalog[['ID', 'LF_WEIGHT']], keys='ID')
cluster_LF_WEIGHT.rename_column('LF_WEIGHT', 'expected_lf_boost')
comparison = join(
    comparison,
    cluster_LF_WEIGHT,
    keys='ID',
    join_type='left',
    metadata_conflicts='silent',
)

lambda_no_lf = col_float(comparison, NO_LF_COL)
lambda_lf = col_float(comparison, LF_COL)
expected_lf_boost = col_float(comparison, 'expected_lf_boost')

if RM_RICHNESS_COL in comparison.colnames:
    lambda_rm = col_float(comparison, RM_RICHNESS_COL)
else:
    lambda_rm = np.full(len(comparison), np.nan)

if Z_COL in comparison.colnames:
    z_cluster = col_float(comparison, Z_COL)
else:
    z_cluster = np.full(len(comparison), np.nan)

with np.errstate(divide='ignore', invalid='ignore'):
    richness_boost = lambda_lf / lambda_no_lf
    boost_residual = richness_boost / expected_lf_boost

valid_cluster = (
    finite_positive(lambda_no_lf)
    & finite_positive(lambda_lf)
    & finite_positive(expected_lf_boost)
    & np.isfinite(richness_boost)
    & np.isfinite(boost_residual)
    & np.isfinite(z_cluster)
)

print('Cluster richness file:', RM_SPEC_FILE)
print('No-LF richness column:', NO_LF_COL)
print('LF richness column:', LF_COL)
print(f'Clusters: {len(comparison):,}')
print(f'Valid comparison clusters: {np.count_nonzero(valid_cluster):,}')
print('Measured boost percentiles:', np.nanpercentile(richness_boost[valid_cluster], [0, 1, 16, 50, 84, 95, 99, 100]))
print('Expected LF boost percentiles:', np.nanpercentile(expected_lf_boost[valid_cluster], [0, 1, 16, 50, 84, 95, 99, 100]))
print('Measured/expected boost percentiles:', np.nanpercentile(boost_residual[valid_cluster], [0, 1, 16, 50, 84, 95, 99, 100]))

display(summarize_by_z(richness_boost[valid_cluster], z_cluster[valid_cluster]))
display(summarize_by_z(expected_lf_boost[valid_cluster], z_cluster[valid_cluster]))
display(summarize_by_z(boost_residual[valid_cluster], z_cluster[valid_cluster]))


In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 5.6))

sc = ax.scatter(
    lambda_no_lf[valid_cluster],
    lambda_lf[valid_cluster],
    c=z_cluster[valid_cluster],
    s=16,
    alpha=0.55,
    cmap='viridis',
    edgecolor='none',
)
lo = np.nanpercentile(np.concatenate([lambda_no_lf[valid_cluster], lambda_lf[valid_cluster]]), 1)
hi = np.nanpercentile(np.concatenate([lambda_no_lf[valid_cluster], lambda_lf[valid_cluster]]), 99)
ax.plot([lo, hi], [lo, hi], color='black', lw=1.2, ls='--', label='1:1')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'No-LF $\lambda_{\rm spec}$')
ax.set_ylabel(r'LF-weighted $\lambda_{\rm spec}$')
ax.legend(frameon=False, loc='upper left')
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label(r'$z_{\rm spec,BCG}$')
style_axis(ax)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'lambda_spec_lf_vs_no_lf.png', dpi=180)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 4.8))

ax.scatter(
    z_cluster[valid_cluster],
    richness_boost[valid_cluster],
    s=12,
    alpha=0.28,
    color='0.25',
    edgecolor='none',
    label='measured richness boost',
)
ax.scatter(
    z_cluster[valid_cluster],
    expected_lf_boost[valid_cluster],
    s=8,
    alpha=0.35,
    color='tab:red',
    edgecolor='none',
    label='expected cluster LF boost',
)

for values, color, marker, label in [
    (richness_boost, 'black', 'o', 'measured median'),
    (expected_lf_boost, 'tab:red', 's', 'expected median'),
]:
    summary_boost = summarize_by_z(values[valid_cluster], z_cluster[valid_cluster])
    zcen = 0.5 * (summary_boost['zlo'].values + summary_boost['zhi'].values)
    med = summary_boost['median'].values
    p16 = summary_boost['p16'].values
    p84 = summary_boost['p84'].values
    ax.errorbar(
        zcen,
        med,
        yerr=[med - p16, p84 - med],
        fmt=marker,
        color=color,
        ecolor=color,
        capsize=0,
        label=label,
        zorder=5,
    )

ax.axhline(1.0, color='black', lw=1.0, ls='--')
ax.set_yscale('log')
ax.set_xlabel(r'$z_{\rm spec,BCG}$')
ax.set_ylabel(r'boost factor')
ax.legend(frameon=False, fontsize=9)
style_axis(ax)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'lambda_spec_lf_boost_vs_redshift.png', dpi=180)
plt.show()


## Measured Boost Versus Expected LF Boost

If the LF correction were applied as a final cluster-level multiplicative factor, the measured richness boost would lie on the 1:1 line below. Scatter away from this line means the current richness pipeline is doing something more complicated than multiplying each cluster richness by its own `w_LF(z_c)`.


In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 5.6))

sc = ax.scatter(
    expected_lf_boost[valid_cluster],
    richness_boost[valid_cluster],
    c=z_cluster[valid_cluster],
    s=18,
    alpha=0.55,
    cmap='viridis',
    edgecolor='none',
)
lo = np.nanpercentile(np.concatenate([expected_lf_boost[valid_cluster], richness_boost[valid_cluster]]), 1)
hi = np.nanpercentile(np.concatenate([expected_lf_boost[valid_cluster], richness_boost[valid_cluster]]), 99)
ax.plot([lo, hi], [lo, hi], color='black', lw=1.2, ls='--', label='1:1')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'expected $w_{\rm LF}(z_c)$')
ax.set_ylabel(r'measured $\lambda_{\rm spec,LF}/\lambda_{\rm spec,noLF}$')
ax.legend(frameon=False, loc='upper left')
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label(r'$z_{\rm spec,BCG}$')
style_axis(ax)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'measured_vs_expected_lf_boost.png', dpi=180)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.0, 5.0), sharey=True)

for ax, y, ylabel, title in [
    (axes[0], lambda_no_lf, r'No-LF $\lambda_{\rm spec}$', 'Before LF weighting'),
    (axes[1], lambda_lf, r'LF-weighted $\lambda_{\rm spec}$', 'After LF weighting'),
]:
    for (zlo, zhi), color in zip(z_bins, z_colors):
        mask = (
            valid_cluster
            & finite_positive(lambda_rm)
            & (z_cluster >= zlo)
            & (z_cluster < zhi)
        )
        ax.scatter(
            lambda_rm[mask],
            y[mask],
            s=14,
            alpha=0.45,
            color=color,
            edgecolor='none',
            label=rf'${zlo}<z<{zhi}$',
        )
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$\lambda_{\rm RM}$')
    ax.set_ylabel(ylabel)
    ax.text(0.04, 0.96, title, transform=ax.transAxes, ha='left', va='top')
    ax.legend(frameon=False, fontsize=9)
    style_axis(ax)

fig.tight_layout()
fig.savefig(PLOT_DIR / 'lambda_spec_vs_lambda_rm_before_after_lf.png', dpi=180)
plt.show()


## Summary Statistics

The table below is a compact check of whether the boost factor increases with redshift. If the LF correction is responsible for removing luminosity incompleteness, the median boost should be larger in the high-redshift bin.


In [ ]:
summary = summarize_by_z(richness_boost[valid_cluster], z_cluster[valid_cluster])
summary['boost_minus_one_percent'] = 100.0 * (summary['median'] - 1.0)
display(summary)

rs, ps = spearmanr(z_cluster[valid_cluster], richness_boost[valid_cluster])
rp, pp = pearsonr(z_cluster[valid_cluster], np.log10(richness_boost[valid_cluster]))
print(f'Spearman(z, boost): r_s={rs:.3f}, p={ps:.3e}')
print(f'Pearson(z, log10 boost): r={rp:.3f}, p={pp:.3e}')
